# Notebook 05 — KPI Computation & Final Tableau Load Preparation
**Project:** FIFA World Cup Analytics | SectionE_g11  
**Purpose:** Compute all KPIs, assemble Tableau-optimized flat export files, and produce the final analytical summary.

In [ ]:
import pandas as pd
import numpy as np
import os

PROCESSED = '../data/processed/'
TABLEAU   = '../tableau/'
os.makedirs(TABLEAU, exist_ok=True)

master  = pd.read_csv(PROCESSED + 'wc_master.csv')
matches = pd.read_csv(PROCESSED + 'wc_matches_clean.csv').drop_duplicates(subset='MatchID')
cups    = pd.read_csv(PROCESSED + 'wc_cups_clean.csv')

print('Data loaded.')

## KPI 1 — Win Rate by Team

In [ ]:
home_df = matches[['Year','Home_Team','Home_Goals','Away_Goals','Match_Result']].copy()
home_df.columns = ['Year','Team','Goals_For','Goals_Against','Match_Result']
home_df['Win']  = (home_df['Match_Result'] == 'Home Win').astype(int)
home_df['Draw'] = (home_df['Match_Result'] == 'Draw').astype(int)
home_df['Loss'] = (home_df['Match_Result'] == 'Away Win').astype(int)

away_df = matches[['Year','Away_Team','Away_Goals','Home_Goals','Match_Result']].copy()
away_df.columns = ['Year','Team','Goals_For','Goals_Against','Match_Result']
away_df['Win']  = (away_df['Match_Result'] == 'Away Win').astype(int)
away_df['Draw'] = (away_df['Match_Result'] == 'Draw').astype(int)
away_df['Loss'] = (away_df['Match_Result'] == 'Home Win').astype(int)

team_match = pd.concat([home_df, away_df], ignore_index=True)

kpi_team = team_match.groupby('Team').agg(
    Matches_Played = ('Win', 'count'),
    Wins           = ('Win', 'sum'),
    Draws          = ('Draw', 'sum'),
    Losses         = ('Loss', 'sum'),
    Goals_For      = ('Goals_For', 'sum'),
    Goals_Against  = ('Goals_Against', 'sum'),
    Tournaments    = ('Year', 'nunique')
).reset_index()

kpi_team['Win_Rate_Pct']   = (kpi_team['Wins'] / kpi_team['Matches_Played'] * 100).round(1)
kpi_team['Goal_Diff']      = kpi_team['Goals_For'] - kpi_team['Goals_Against']
kpi_team['Goals_Per_Match']= (kpi_team['Goals_For'] / kpi_team['Matches_Played']).round(2)
kpi_team['Points']         = kpi_team['Wins'] * 3 + kpi_team['Draws']  # Modern point system

print('KPI 1 — Win Rate by Team (Top 10):')
print(kpi_team[kpi_team['Matches_Played'] >= 10].nlargest(10, 'Win_Rate_Pct')
      [['Team','Matches_Played','Wins','Win_Rate_Pct','Goal_Diff','Tournaments']].to_string(index=False))

## KPI 2 — Goals per Match by Year & Stage

In [ ]:
kpi_goals_year = matches.groupby('Year').agg(
    Avg_Goals_Per_Match = ('Total_Goals', 'mean'),
    Total_Goals         = ('Total_Goals', 'sum'),
    Matches             = ('MatchID', 'count'),
    Avg_Attendance      = ('Attendance', 'mean')
).reset_index().round(2)

kpi_goals_stage = matches.groupby('Stage_Std').agg(
    Avg_Goals_Per_Match = ('Total_Goals', 'mean'),
    Total_Matches       = ('MatchID', 'count')
).reset_index().round(2)

print('KPI 2 — Goals by Year:')
print(kpi_goals_year.to_string(index=False))
print('\nKPI 2 — Goals by Stage:')
print(kpi_goals_stage.to_string(index=False))

## KPI 3 — Home Advantage Index

In [ ]:
overall_win_rate = (matches['Match_Result'] == 'Home Win').mean() * 100

kpi_home_adv = matches.groupby('Year').apply(
    lambda df: pd.Series({
        'Home_Win_Rate': (df['Match_Result'] == 'Home Win').mean() * 100,
        'Avg_Home_Goals': df['Home_Goals'].mean(),
        'Avg_Away_Goals': df['Away_Goals'].mean(),
        'Home_Advantage_Index': (df['Home_Goals'].mean() - df['Away_Goals'].mean())
    })
).reset_index().round(2)

print(f'Overall Home Win Rate (all years): {overall_win_rate:.1f}%')
print('\nKPI 3 — Home Advantage Index by Year:')
print(kpi_home_adv.to_string(index=False))

## KPI 4 — Player Goal Contribution Rate

In [ ]:
kpi_player = master.groupby(['Player_Name', 'Team_Initials']).agg(
    Matches_Played = ('MatchID', 'nunique'),
    Goals_Scored   = ('Goals_Scored', 'sum'),
    Yellow_Cards   = ('Yellow_Cards', 'sum'),
    Red_Cards      = ('Red_Cards', 'sum'),
    Is_Substitute  = ('Is_Substitute', 'sum')
).reset_index()

kpi_player['Goals_Per_Match'] = (kpi_player['Goals_Scored'] / kpi_player['Matches_Played']).round(3)
kpi_player = kpi_player[kpi_player['Matches_Played'] >= 3]

print('KPI 4 — Top Goal Scorers (min 3 matches):')
print(kpi_player.nlargest(15, 'Goals_Scored')[['Player_Name','Team_Initials','Matches_Played','Goals_Scored','Goals_Per_Match']].to_string(index=False))

## KPI 5 — Attendance Growth Rate

In [ ]:
kpi_attendance = cups[['Year','Attendance']].copy()
kpi_attendance = kpi_attendance.sort_values('Year')
kpi_attendance['Attendance_Growth_Pct'] = kpi_attendance['Attendance'].pct_change() * 100
kpi_attendance['Avg_Per_Match'] = kpi_attendance['Attendance'] / cups['Matches_Played']

print('KPI 5 — Attendance Growth:')
print(kpi_attendance.round(1).to_string(index=False))

## KPI 6 — Team Consistency Score

In [ ]:
team_years = team_match.groupby('Team')['Year'].nunique().reset_index()
team_years.columns = ['Team', 'Tournaments_Appeared']

kpi_consistency = kpi_team.merge(team_years, on='Team')
kpi_consistency['Consistency_Score'] = (
    kpi_consistency['Win_Rate_Pct'] * 0.5 +
    kpi_consistency['Tournaments_Appeared'] * 2
).round(1)

print('KPI 6 — Team Consistency Score (Top 15):')
print(kpi_consistency.nlargest(15, 'Consistency_Score')
      [['Team','Tournaments_Appeared','Win_Rate_Pct','Consistency_Score']].to_string(index=False))

## Final Tableau Export Files

In [ ]:
# 1. Master flat file for Tableau (player-match level)
master.to_csv(TABLEAU + 'tableau_master.csv', index=False)

# 2. Match-level summary for Tableau
matches.to_csv(TABLEAU + 'tableau_matches.csv', index=False)

# 3. KPI — Team performance
kpi_team.to_csv(TABLEAU + 'kpi_team_performance.csv', index=False)

# 4. KPI — Goals by year and stage
kpi_goals_year.to_csv(TABLEAU + 'kpi_goals_by_year.csv', index=False)
kpi_goals_stage.to_csv(TABLEAU + 'kpi_goals_by_stage.csv', index=False)

# 5. KPI — Home advantage by year
kpi_home_adv.to_csv(TABLEAU + 'kpi_home_advantage.csv', index=False)

# 6. KPI — Player goal contribution
kpi_player.to_csv(TABLEAU + 'kpi_player_goals.csv', index=False)

# 7. KPI — Attendance
kpi_attendance.to_csv(TABLEAU + 'kpi_attendance.csv', index=False)

# 8. KPI — Consistency
kpi_consistency.to_csv(TABLEAU + 'kpi_team_consistency.csv', index=False)

print('Tableau export files written:')
for f in sorted(os.listdir(TABLEAU)):
    if f.endswith('.csv'):
        rows = len(pd.read_csv(TABLEAU + f))
        size = os.path.getsize(TABLEAU + f) // 1024
        print(f'  {f:<40} — {rows:,} rows, {size} KB')

## Business Recommendations

Based on all analysis performed across notebooks 03–05, here are 5 actionable recommendations for football federations:

| # | Recommendation | Evidence |
|---|---------------|----------|
| 1 | **Invest in first-half dominance training** — half-time score predicts 70%+ of final outcomes. Squads that lead at half-time win >80% of matches. | Linear regression R² ≈ 0.70; logistic model accuracy ≈ 75% |
| 2 | **Mandatory penalty training for knockout squads** — penalty shootouts are statistically concentrated in Round of 16 and Quarter-Finals. Chi-square confirms non-random stage distribution. | Chi-square p < 0.001 |
| 3 | **Adjust defensive strategy by stage** — reduce attacking risk in knock-out rounds where avg goals drop ~30% vs group stage. Keep expansive play for group stage point accumulation. | Group avg 3.1 goals vs knockout avg 2.2 |
| 4 | **Exploit home advantage when hosting** — host nations win 8–12% more matches than their historical average. Squad preparation should double down on crowd-engagement drills for future hosting bids. | Paired t-test p < 0.001; HAI analysis |
| 5 | **Track team consistency over individual star quality** — teams appearing in 15+ tournaments with >50% win rates (Brazil, Germany) outperform teams that peak in one edition. Long-term federation development beats short-term star recruitment. | Consistency Score KPI |

---
**All analysis complete. Export files ready for Tableau Public dashboard construction.**